In [ ]:
import tensorflow as tf
import numpy as np
import cv2

In [ ]:
CLASS_NAMES = ["CSC", "Normal", "PCV", "VKH"]
SCAN_TYPES = {
    "radial":12,
    "raster":25
}

### Load Models

In [ ]:
tsa_fe_model = tf.keras.models.load_model("../../inference_service/src/models/tsa_feature_extractor.h5")
tsa_lstm_model = tf.keras.models.load_model("../../inference_service/src/models/tsa_lstm.keras")

### Load Data

In [ ]:
radial_sequence = []
raster_sequence = []
for st, num_images in SCAN_TYPES.items():
    for i in range(1, num_images + 1):
        img = cv2.imread(f"../data/preprocessed/{st}/{i}.png")
        if st == "radial":
            radial_sequence.append(img)
        else:
            raster_sequence.append(img)

### Feature Extraction

In [ ]:
radial_features = tsa_fe_model.predict(np.array(radial_sequence))
raster_features = tsa_fe_model.predict(np.array(raster_sequence))
radial_features = np.expand_dims(radial_features, axis=0)
raster_features = np.expand_dims(raster_features, axis=0)
input_features = [radial_features, raster_features]

### Prediction

In [ ]:
output = tsa_lstm_model.predict([radial_features, raster_features])

scores = output[0].tolist()
pred_class = CLASS_NAMES[np.argmax(scores)]

In [ ]:
print(f"Predicted class: {pred_class}")
print(f"Class scores: {scores}")